# Chord progressions and Spotify audio features

This notebook explores the tracks matched between Chordonomicon and Spotify in `new_dataset.csv`.
In the supplied file, 13,579 joined rows represent 9,079 Spotify tracks. Release decade is missing for
566 tracks and main genre is missing for 1,495. Each chart uses the tracks with the fields it needs.
The matched tracks describe this dataset's coverage.

Run all cells from the project root or from the `young` directory with pandas and Plotly available.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

data_path = Path("new_dataset.csv")
if not data_path.is_file():
    data_path = Path("../new_dataset.csv")

raw = pd.read_csv(data_path)
import plotly.io as pio

pio.templates.default = "plotly_white"

# Give each distinct transcription one vote within its track.
transcriptions = raw[["track_id", "chords"]].dropna().drop_duplicates().copy()
transcriptions["chord_count"] = transcriptions["chords"].map(
    lambda sequence: len({token for token in sequence.split() if not token.startswith("<")})
)
chord_counts = transcriptions.groupby("track_id")["chord_count"].median()
popularity = (
    raw[["track_id", "popularity"]].drop_duplicates()
    .groupby("track_id")["popularity"].median()
)

tracks = raw.drop_duplicates("track_id").set_index("track_id").copy()
tracks["chord_count"] = chord_counts
tracks["popularity"] = popularity
tracks["decade_label"] = tracks["decade"].astype("Int64").astype("string") + "s"

genre_order = tracks["main_genre"].value_counts().index.tolist()
feature_labels = {
    "danceability": "Danceability",
    "energy": "Energy",
    "acousticness": "Acousticness",
    "valence": "Valence",
    "speechiness": "Speechiness",
    "instrumentalness": "Instrumentalness",
    "liveness": "Liveness",
    "tempo": "Tempo (BPM)",
    "loudness": "Loudness (dB)",
}
display(pd.Series({
    "Joined rows": len(raw),
    "Unique Spotify tracks": len(tracks),
    "Tracks missing release decade": tracks["decade"].isna().sum(),
    "Tracks missing main genre": tracks["main_genre"].isna().sum(),
}).to_frame("Count"))

,Count
Joined rows,13579
Unique Spotify tracks,9079
Tracks missing release decade,566
Tracks missing main genre,1495


Chord diversity is the number of distinct chord labels after removing section markers such as
`<verse_1>`. Labels are used as written, so different spellings count separately. If a track has
multiple distinct transcriptions, its value is the median across them. If a track has multiple
distinct Spotify popularity scores, its value is their median.

## 1. Which release decades does the dataset cover?

Compare the counts before interpreting genre or release-era patterns. Tracks without a release decade
appear in the Unknown bar.

In [2]:
decade_counts = (
    tracks["decade_label"].fillna("Unknown").value_counts().sort_index()
    .rename_axis("Release decade").reset_index(name="Tracks")
)
fig = px.bar(
    decade_counts, x="Release decade", y="Tracks",
    title="Matched Spotify tracks by release decade",
)
fig.update_traces(hovertemplate="%{x}<br>%{y:,} tracks<extra></extra>")
fig.show()

## 2. How does chord diversity vary by genre?

Each box shows the median and middle half of tracks in a Chordonomicon main genre. Genres are
ordered by their number of tracks. The sample size appears in each genre label.

In [3]:
genre_tracks = tracks.dropna(subset=["main_genre", "chord_count"]).copy()
genre_sizes = genre_tracks["main_genre"].value_counts()
genre_labels = {genre: f"{genre} (n={genre_sizes[genre]:,})" for genre in genre_order}
genre_tracks["Genre"] = genre_tracks["main_genre"].map(genre_labels)

fig = px.box(
    genre_tracks, x="Genre", y="chord_count", points="outliers",
    category_orders={"Genre": list(genre_labels.values())},
    labels={"chord_count": "Distinct chord labels per track"},
    title="Chord diversity by main genre",
)
fig.update_xaxes(tickangle=-35)
fig.show()

## 3. How do typical Spotify audio features differ by genre?

Each cell shows the median Spotify score for a genre. The features in this chart all use a
zero-to-one scale.

In [4]:
profile_features = list(feature_labels)[:7]
genre_profile = (
    tracks.groupby("main_genre")[profile_features].median()
    .reindex(genre_order).rename(columns=feature_labels)
)
fig = px.imshow(
    genre_profile, text_auto=".2f", aspect="auto", zmin=0, zmax=1,
    color_continuous_scale="Blues",
    labels={"x": "Spotify audio feature", "y": "Main genre", "color": "Median score"},
    title="Spotify audio profiles by main genre",
)
fig.show()

## 4. How do energy and acousticness relate to chord diversity?

Each dot represents one track. Color shows its chord diversity. Hover to see the track name and artist.

In [5]:
fig = px.scatter(
    tracks, x="acousticness", y="energy", color="chord_count",
    hover_name="track_name", hover_data={"artists": True, "main_genre": True},
    labels={
        "acousticness": "Spotify acousticness", "energy": "Spotify energy",
        "chord_count": "Distinct chords",
    },
    title="Energy and acousticness across tracks",
    color_continuous_scale="Viridis", opacity=0.55, render_mode="webgl",
)
fig.update_traces(marker_size=5)
fig.show()

## 5. Which audio features move with chord diversity?

Spearman correlation measures how consistently two variables rise or fall together in rank.
Estimates pool the matched tracks across genres and release decades.

In [6]:
correlations = (
    tracks[["chord_count", *feature_labels]].corr(method="spearman")
    ["chord_count"].drop("chord_count").rename(index=feature_labels).sort_values()
)
fig = px.bar(
    x=correlations.values, y=correlations.index, orientation="h",
    labels={"x": "Spearman rank correlation", "y": "Spotify audio feature"},
    title="Audio feature correlations with chord diversity",
)
fig.update_traces(hovertemplate="%{y}<br>Spearman correlation: %{x:.3f}<extra></extra>")
fig.add_vline(x=0, line_color="#64748B", line_width=1)
fig.update_xaxes(range=[-1, 1])
fig.show()

## 6. How does Spotify popularity vary across chord diversity groups?

Dots show median popularity and bars span the middle half of scores. Hover for group sample sizes.
The bins include fractional values from tracks with multiple transcriptions.

In [7]:
popularity_groups = tracks.assign(
    chord_group=pd.cut(
        tracks["chord_count"], bins=[0, 5, 10, 15, float("inf")],
        labels=["Up to 5", "Over 5 to 10", "Over 10 to 15", "Over 15"],
        include_lowest=True,
    )
)
popularity_summary = popularity_groups.groupby("chord_group", observed=True)["popularity"].agg(
    tracks="count",
    median="median",
    lower_quartile=lambda scores: scores.quantile(0.25),
    upper_quartile=lambda scores: scores.quantile(0.75),
)

fig = go.Figure(go.Scatter(
    x=popularity_summary.index.tolist(),
    y=popularity_summary["median"],
    mode="markers",
    marker={"size": 10, "color": "#2563EB"},
    error_y={
        "type": "data", "symmetric": False,
        "array": popularity_summary["upper_quartile"] - popularity_summary["median"],
        "arrayminus": popularity_summary["median"] - popularity_summary["lower_quartile"],
    },
    customdata=popularity_summary[["tracks", "lower_quartile", "upper_quartile"]],
    hovertemplate=(
        "%{x} distinct chords<br>Median popularity: %{y:.1f}<br>"
        "Middle 50%: %{customdata[1]:.1f} to %{customdata[2]:.1f}<br>"
        "%{customdata[0]:,.0f} tracks<extra></extra>"
    ),
))
fig.update_layout(
    title="Spotify popularity by chord diversity",
    xaxis_title="Distinct chord labels per track",
    yaxis_title="Spotify popularity score",
)
fig.update_yaxes(range=[0, 100])
fig.show()